# 03 - Projected kernel: 1-RDM features (week 1)
Owner: **Shams**

The projected kernel compares samples through their **one-qubit reduced density matrices** (1-RDMs). Each 1-RDM is fixed by three numbers per qubit:

$$\rho_k(x)=\tfrac{1}{2}\big(I+\langle X_k\rangle_x X+\langle Y_k\rangle_x Y+\langle Z_k\rangle_x Z\big)$$

Measurement: $Z$ directly; $X$ after $H$; $Y$ after $S^\dagger$, then $H$. One circuit per basis gives that Pauli for all qubits, so each sample costs **3 circuits** (the fidelity kernel costs one circuit per *pair*).

Week 1 (this notebook): extract the features, exact and sampled, and check them. Week 2: the kernel itself and its tuning.

In [1]:
import sys, json
sys.path.insert(0, "..")
import numpy as np
from src.common import get_split, make_preprocessor, encoding_circuit, result_path, SHOTS
from src.projected import pauli_features_exact, pauli_features_sampled, rdm_checks, rdms_from_features

## 1. Settings (same data, circuit and encoding scale as notebook 02)

In [2]:
SEED     = 0
N_QUBITS = 4
LAYERS   = 2
ALPHA    = 1.0     # best alpha from notebook 02; the projected kernel tunes its own in week 2

## 2. Data and encoding angles

In [3]:
Xtr, ytr, Xte, yte = get_split(SEED)
tf = make_preprocessor(N_QUBITS, ALPHA)(Xtr)    # fitted on training data only
Atr, Ate = tf(Xtr), tf(Xte)
enc = encoding_circuit(N_QUBITS, LAYERS)
print("train", Atr.shape, "test", Ate.shape)

train (80, 4) test (40, 4)


## 3. Exact Pauli features (ideal reference)

In [4]:
Fe_tr, Fe_te = pauli_features_exact(enc, Atr), pauli_features_exact(enc, Ate)
print("shape (samples, qubits, [X, Y, Z]):", Fe_tr.shape)
print("\nfirst training sample:")
print(f"{'qubit':>6}{'<X>':>9}{'<Y>':>9}{'<Z>':>9}")
for k in range(N_QUBITS):
    print(f"{k:>6}" + "".join(f"{v:>9.3f}" for v in Fe_tr[0, k]))
print(f"\nmax |<Y>| over all samples and qubits: {np.abs(Fe_tr[..., 1]).max():.1e}  (theory: 0)")
print("1-RDM checks:", rdm_checks(Fe_tr))

shape (samples, qubits, [X, Y, Z]): (80, 4, 3)

first training sample:
 qubit      <X>      <Y>      <Z>
     0    0.149    0.000    0.953
     1   -0.047    0.000    0.644
     2    0.025    0.000    0.791
     3    0.132    0.000    0.635

max |<Y>| over all samples and qubits: 0.0e+00  (theory: 0)
1-RDM checks: {'max_trace_error': 0.0, 'max_bloch_length': 0.9999618134706878, 'min_eigenvalue': 1.909326465609988e-05}


**Why $\langle Y\rangle = 0$:** the circuit uses only $R_y$ and CNOT gates, which have real matrices, so the state has real amplitudes. For any real state the $Y$ expectation is zero. We still measure $Y$ as a consistency check under shot noise and device noise.

## 4. Sampled Pauli features (3 circuits per sample, as on hardware)

In [5]:
Fs_tr = pauli_features_sampled(enc, Atr, shots=SHOTS, seed=SEED)
Fs_te = pauli_features_sampled(enc, Ate, shots=SHOTS, seed=SEED + 1)
err = np.abs(Fs_tr - Fe_tr)
print(f"shots per circuit: {SHOTS}")
print(f"|sampled - exact|: mean {err.mean():.4f}, max {err.max():.4f}")
print(f"expected shot noise per value: about {1/np.sqrt(SHOTS):.4f} (1/sqrt(shots))")
print(f"sampled <Y>: mean {Fs_tr[..., 1].mean():+.4f}, max |.| {np.abs(Fs_tr[..., 1]).max():.4f}  (pure shot noise around 0)")
print("1-RDM checks (sampled):", rdm_checks(Fs_tr))

shots per circuit: 2048
|sampled - exact|: mean 0.0152, max 0.0742
expected shot noise per value: about 0.0221 (1/sqrt(shots))
sampled <Y>: mean +0.0004, max |.| 0.0742  (pure shot noise around 0)
1-RDM checks (sampled): {'max_trace_error': 0.0, 'max_bloch_length': 1.0012818703597568, 'min_eigenvalue': -0.0006409351798783974}


## 5. Resources and save

In [6]:
n_samples = len(Atr) + len(Ate)
print(f"circuits: 3 per sample x {n_samples} samples = {3 * n_samples}")
print(f"shots: {3 * n_samples * SHOTS:,}")
fid_pairs = len(Atr) * (len(Atr) - 1) // 2 + len(Ate) * len(Atr)
print(f"(fidelity kernel needs {fid_pairs:,} circuits for the same data)")

np.save(result_path("features", "pauli_exact_train", SEED, N_QUBITS, "npy"), Fe_tr)
np.save(result_path("features", "pauli_exact_test", SEED, N_QUBITS, "npy"), Fe_te)
np.save(result_path("features", "pauli_sampled_train", SEED, N_QUBITS, "npy"), Fs_tr)
np.save(result_path("features", "pauli_sampled_test", SEED, N_QUBITS, "npy"), Fs_te)
print("saved to results/features/")

circuits: 3 per sample x 120 samples = 360
shots: 737,280
(fidelity kernel needs 6,360 circuits for the same data)
saved to results/features/
